# Inspect a campaign dataset

Check integrity, provenance, reproduction and model-ready arrays.
Also runnable as `python examples/04_dataset.py`.

In [1]:

import sys
from pathlib import Path

examples_dir = next(
    (root / "examples" for root in (Path.cwd(), *Path.cwd().parents)
     if (root / "examples" / "rc_study.py").is_file()),
    None,
)
if examples_dir is not None:
    sys.path.insert(0, str(examples_dir))

In [2]:
from rc_study import WORKERS, campaign  # noqa: E402

from spicefault import Dataset  # noqa: E402

## Ensure the campaign dataset exists

This notebook can run independently; it creates the dataset if campaign 03 has not run.

In [3]:
if __name__ == "__main__":
    campaign().run(workers=WORKERS, chunk=200, progress=False)

    # everything below needs only the folder
    dataset = Dataset(campaign().out_dir)
    print(dataset)
    print("files:", sorted(p.name for p in dataset.path.iterdir()))
    print("problems found by verify():", dataset.verify())

Dataset('/Users/telmomm/Desktop/spicefault/examples/output/rc_campaign': 660 samples, 18 faults, 5 measurements)
files: ['circuit.cir', 'manifest.json', 'metadata.json', 'samples.parquet', 'waveforms.npy']
problems found by verify(): []


## Trace a sample back to its simulation

In [4]:
if __name__ == "__main__":
    sample_id = 350
    print(f"\nprovenance of sample {sample_id}:")
    for key, value in dataset.provenance(sample_id).to_dict().items():
        print(f"  {key}: {value}")

    print(f"\nthe netlist that was simulated for sample {sample_id}:")
    print(dataset.netlist(sample_id))


provenance of sample 350:
  sample_id: 350
  seed: 42
  seeding: content
  seed_key: 2495436275234074564/10
  circuit: rc_lowpass
  netlist_sha256: f0efb2ba450dd22f373836aee5b675e21532f9f2a960feef89bb86593112fa0f
  fault_id: R1:parametric:-0.5
  fault_type: parametric
  fault_location: R1
  fault_magnitude: -0.5
  fault_severity: 0.5
  condition: nominal
  status: SUCCESS
  message: 
  simulator: ngspice
  simulator_version: ngspice-44.2
  spicefault_version: 0.1.0
  parameters: {'p_R1_value': 10037.886198176157, 'p_C1_value': 1.0473527298915118e-07, 'p_R2_value': 9978.972456385483}

the netlist that was simulated for sample 350:
RC low-pass with a resistive load
* 10 kOhm source resistance, 100 nF, 10 kOhm load: 0.5 V/V at DC, corner at 318 Hz
V1 in 0 dc 1 ac 1 pulse(0 1 1m 1u 1u 5m 20m)
R1 in out 5018.943099088078
C1 out 0 1.0473527298915118e-07
R2 out 0 9978.972456385483
.end



## Reproduce measurements and prepare arrays

In [5]:
if __name__ == "__main__":
    print("simulating 20 samples again and comparing with what is stored:")
    report = dataset.reproduce(n=20)
    print(report.head().to_string(index=False))
    print("largest difference in the measurements:", report["max_abs_diff"].max())
    print("largest difference in the waveforms:   ", report["waveform_abs_diff"].max())

    x, y = dataset.to_ml(target="fault_location")
    print(f"\nfor a classifier: X {x.shape}, labels {sorted(set(y))}")
    waves, kind = dataset.to_ml(target="fault_type", waveforms=True)
    print(f"or the waveforms: X {waves.shape}, labels {sorted(set(kind))}")

simulating 20 samples again and comparing with what is stored:
 sample_id  definition  parameters  status  max_abs_diff  max_rel_diff  waveform_abs_diff
        10        True        True    True           0.0           0.0                0.0
        26        True        True    True           0.0           0.0                0.0
        48        True        True    True           0.0           0.0                0.0
       113        True        True    True           0.0           0.0                0.0
       173        True        True    True           0.0           0.0                0.0
largest difference in the measurements: 0.0
largest difference in the waveforms:    0.0

for a classifier: X (660, 5), labels ['', 'C1', 'R1', 'R2']
or the waveforms: X (660, 100), labels ['healthy', 'open_circuit', 'parametric', 'short_circuit']
